# 00 · Setup check

Run this first, on Colab, Kaggle or your laptop. In under five minutes it confirms that:

1. the repository and its dependencies are installed,
2. the shared configuration loads (same settings for every model, as the assignment requires),
3. all **five networks** we compare can be built and produce one score per DR grade,
4. pretrained ImageNet weights can be downloaded (needed for transfer learning).

No dataset is needed yet; the data arrives in `01_eda.ipynb`.

In [ ]:
# Locate the repository, or clone it when this runs in a fresh Colab/Kaggle runtime.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/MOUATEAlaaeddine/dr-transfer-learning.git"

here = Path.cwd()
if (here / "src" / "drtl").exists():
    root = here
elif (here.parent / "src" / "drtl").exists():          # opened from notebooks/
    root = here.parent
else:                                                    # fresh cloud runtime
    if not Path("dr-transfer-learning").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    root = Path("dr-transfer-learning").resolve()

os.chdir(root)
sys.path.insert(0, str(root / "src"))

# Colab and Kaggle already ship PyTorch; pip only installs what is missing (usually timm).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Repository:", root)

## Shared configuration

Every experiment reads `configs/default.yaml`. A notebook may override a value for one run
(e.g. a smaller batch size for VGG19), but never edits the shared defaults silently.

In [ ]:
from drtl.config import load_config

config = load_config()
print("Grades:", config["data"]["class_names"])
print("Image size:", config["data"]["image_size"], "| test fraction:", config["data"]["test_fraction"],
      "| CV folds:", config["data"]["n_folds"])
print("Strategies:", list(config["strategies"]))

## Reproducibility and hardware

Seed 42 for every run. Training uses CUDA on Colab/Kaggle; on a Mac the code falls back to MPS, otherwise CPU.

In [ ]:
import torch
from drtl.utils import get_device, set_seed

set_seed(config["seed"])
device = get_device()
print("PyTorch", torch.__version__, "| device:", device)

## The five networks

Four ImageNet-pretrained backbones of very different sizes, plus a small CNN trained from scratch
as the baseline. Here they are built **without** pretrained weights (fast, offline) and fed a dummy
batch of two 224×224 images. Each must return 5 scores, one per grade.

The parameter counts below are measured, not estimated; they go into the results table of the paper.

In [ ]:
import pandas as pd
from drtl.models import build_model, count_parameters

num_classes = len(config["data"]["class_names"])
size = config["data"]["image_size"]
dummy_batch = torch.randn(2, 3, size, size)

rows = []
for key, timm_name in config["models"].items():
    model = build_model(timm_name, num_classes, pretrained=False).eval()
    with torch.no_grad():
        logits = model(dummy_batch)
    assert logits.shape == (2, num_classes), key
    rows.append({"model": key, "timm name": timm_name,
                 "parameters (M)": round(count_parameters(model)["total"] / 1e6, 2),
                 "output shape": tuple(logits.shape)})

pd.DataFrame(rows)

## Pretrained weights

Transfer learning starts from ImageNet weights. We download the smallest backbone (MobileNetV4)
to confirm the runtime can reach the weight hub.

In [ ]:
pretrained = build_model(config["models"]["mobilenetv4"], num_classes, pretrained=True).eval()
with torch.no_grad():
    print("MobileNetV4 with ImageNet weights → output", tuple(pretrained(dummy_batch).shape))
print("Setup check passed.")